<a href="https://colab.research.google.com/github/Lulu035/machine-learning-roadmap/blob/master/31Oct_Transfer_Learning_Freezing_Pretrained_model_efficientnet_b0_feature_Layers_update_output_layer_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, zipfile
from pathlib import Path

In [3]:
# 挂载 Google Drive ：
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Save in Google Drive:
data_path = Path('/content/drive/MyDrive/colabnotebooks/pytorchlulu')

# 下载和解压前， 提前创建目录：
data_path.mkdir(parents=True, exist_ok=True)


#下载ZIP 包到 根目录：
zip_file = data_path / "pizza_steak_sushi.zip"

url = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"
r = requests.get(url, timeout=60) # 用 request.get() 向这个 URL 发起请求，拿到相应对象 r
r.raise_for_status() # 如果服务器返回的状态码不是 200（比如 404、500），这里会抛出 HTTPError，从而提前失败，避免你把错误页面当成 zip 保存下来。
with open(zip_file, "wb") as f: # 以二进制状态打开ZIP：
    print ("Downloading and Open ZIP file... ")
    f.write(r.content) # 把全部字节写入文件-pytorchlulu.

# 把ZIP 压缩包解压到指定文件夹：
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    print("unzipping data to : ", data_path)
    zip_ref.extractall(data_path)

print ("Done ! ", data_path.resolve())

unzipping data to :  /content/drive/MyDrive/colabnotebooks/pytorchlulu
Done !  /content/drive/MyDrive/colabnotebooks/pytorchlulu


In [5]:
print (zip_ref.namelist()[:10])

['test/', 'train/', 'test/pizza/', 'test/steak/', 'test/sushi/', 'test/steak/296375.jpg', 'test/steak/354513.jpg', 'test/steak/690177.jpg', 'test/steak/1882831.jpg', 'test/steak/894825.jpg']


In [6]:
#删除压缩包：
zip_file.unlink()

In [7]:
# Setup train and testing paths：
''' 后面要把训练集和测试/验证集分开喂给 Dataset/DataLoader，并且通常会对它们用不同的变换与策略，
用于 torchvision.datasets.ImageFolder(train_dir, ...) / ImageFolder(test_dir, ...)

对 train 加数据增强（随机裁剪、翻转），对 test 只做确定性的 resize/normalise

防止数据泄漏（评估时绝不看训练数据）， 统计各 split 的样本数、计算准确率等'''
train_dir = data_path / "train"
test_dir = data_path / "test"

train_dir, test_dir

(PosixPath('/content/drive/MyDrive/colabnotebooks/pytorchlulu/train'),
 PosixPath('/content/drive/MyDrive/colabnotebooks/pytorchlulu/test'))

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [10]:
import torch
import torchvision
from torchvision import transforms

Convert Image into Tensors:
把多个图片变换按顺序 ‘串起来’：- composition of transforms:
torchvision.Compose([...]}

In [ ]:
# Manually Create :

# 增强与标准化：

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
    normalize                         # 1. 把PIL 图片变成 张量 【C, H,W]. 把0-255 像素缩放到数值范围 【0，1】 2. Min-Max : 归一化： x/255:

])

# Test Set 去掉 随机性：
test_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    normalize
])

# 1. 创建 dataset and DataLoader : Use datasets.ImageFolder() to load in data:

In [ ]:
from torchvision import datasets # 分类任务 + 按文件夹分好：
from torch.utils.data import DataLoader # Moved import from line 2 to line 3 for clarity

# torchvision.datasets.ImageFolder:
train_data = datasets.ImageFolder(train_dir, transform=train_transform)
test_data = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader( train_data, batch_size=32, num_workers=2, shuffle=True)
test_loader = DataLoader( test_data, batch_size=32, num_workers=2, shuffle=False) # Added missing comma here

train_loader
test_loader

In [ ]:
# Check the shape:
img, label = next(iter(train_loader))
img.shape, label.shape

(torch.Size([32, 3, 64, 64]), torch.Size([32]))

In [15]:
# Setup device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


Auto Create :

In [16]:
# Auto Create :
# when use a pre-trained model, custom data must be prepared the same way as the original training data went into model:
# weights = torchvision.models.EfficientNet_B0_Weights.Default:


In [17]:
# get a set of pretrained model weights : 选择权重：
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
# 获取在 ImageNet 上 训练 EfficientNet_B0_weights 的数据变换：


In [18]:
# Fit in Model：
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

In [21]:
!pip install torchinfo

In [22]:
# Import torchinfo
from torchinfo import summary

In [23]:
# get model infor:
# Print a summary using torchinfo (uncomment for actual output)
# 对于 efficientnet_b0 的情况，输入大小是 (batch_size, 3, 224, 224):
summary(model=model,
        input_size=(32, 3, 224, 224), # make sure this is "input_size", not "input_shape"
        # col_names=["input_size"], # uncomment for smaller output
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

先把特征提取层全冻结，看一眼当前模型（还没改头），然后把分类头重建成你任务的类别数


In [24]:
print(r'''EfficientNet-B0 结构拆分

model.features（特征层）：

开头的 stem：Conv2d + BN + SiLU

一串 MBConv 块（深度可分离卷积、Squeeze-Excitation 等）

最后到一个 1280 通道的高层特征

model.avgpool（自适应池化成 1×1）

model.classifier（分类头）：

Dropout(p=0.2)

Linear(1280 → num_classes)''')

EfficientNet-B0 结构拆分

model.features（特征层）：

开头的 stem：Conv2d + BN + SiLU

一串 MBConv 块（深度可分离卷积、Squeeze-Excitation 等）

最后到一个 1280 通道的高层特征

model.avgpool（自适应池化成 1×1）

model.classifier（分类头）：

Dropout(p=0.2)

Linear(1280 → num_classes)


In [25]:
# 冻结特征层： 卷积块的权重在训练中不更新 （ 不计算梯度）： 只训练后面的 classifier： 分类头
# 典型"特征提取"式 Transfer Learning。
# 如何确定哪些是特征层？ ：
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [28]:
# 冻结特征层： 卷积块的权重在训练中不更新 （ 不计算梯度）： 只训练后面的 classifier： 分类头
# Freeze all base layers in the "feature" section ( feature extractor) by setting required_grad = False
for p in model.features.parameters():
    p.requires_grad = False
print("Now Feature Extractor Layers frozen")

Now Feature Extractor Layers frozen


In [30]:
# Adjust the output - the classifer portion of the pre-trained model :
# pretrained model : out_features=1000  , but we only have three : pizza, steak, sushi)
print(r'''(classifier): Sequential(
    (0): Dropout(p=0.2, inplace=True)
    (1): Linear(in_features=1280, out_features=1000, bias=True)
  )''')

(classifier): Sequential(
    (0): Dropout(p=0.2, inplace=True)
    (1): Linear(in_features=1280, out_features=1000, bias=True)
  )


In [35]:
# Recreate the classifier layer and seed it to the target device
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2 ,inplace = True),
    torch.nn.Linear(in_features=1280, out_features=3, bias=True)).to(device)

print ("now output layer - classifier layer updated !")

now output layer - classifier layer updated !
